# Video Transformation

This notebook will be where the videos have their frames extracted, and then run through a backbone to generate feature vectors. The trainin, validation, and testing videos will all be done here.

In [ ]:
# File paths

ROOT = "./HD-EPIC"
FEATURES = f"{PATH}/Features"

Mounted at /content/drive


In [ ]:
!pip install -q av transformers

import os, glob, time, collections, json, random, ast
from pathlib import Path
import numpy as np
import torch
import av
import json
import pandas as pd
from transformers import VideoMAEModel

VIDEO_DIR = f"{ROOT}/Videos"             # <-- EDIT: folder that contains the .mp4 files (searched recursively)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BACKBONE_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"
CLIP_FRAMES = 16        # frames per clip (what VideoMAE-base expects)
SAMPLE_FPS = 8          # 16 frames at 8 fps = a 2 second clip
PREDICTION_HORIZON = 30.0
CLIP_STRIDE = 1.0       # one clip per second of video
SIZE = 224

backbone = VideoMAEModel.from_pretrained(BACKBONE_NAME).to(device).eval()
FEATURE_DIM = backbone.config.hidden_size
print("Feature dim:", FEATURE_DIM)

# ImageNet mean/std, as used by the VideoMAE image processor
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 1, 3, 1, 1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 21.6 MB/s eta 0:00:00


config.json:   0%|          | 0.00/22.9k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status     |  | 
------------------+------------+--+-
fc_norm.weight    | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 
fc_norm.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Feature dim: 768


In [ ]:
# Loading in annotations from the HD-EPIC dataset itself

with open(f"{ROOT}/Annotations/scene-and-object-movements/assoc_info.json") as f:
    assoc_info = json.load(f)

with open(f"{ROOT}/Annotations/scene-and-object-movements/mask_info.json") as f:
    mask_info = json.load(f)

with open(f"{ROOT}/Annotations/eye-gaze-priming/priming_info.json") as f:
    priming_info = json.load(f)


noun_classes = pd.read_csv(
    f"{ROOT}/Annotations/narrations-and-action-segments/HD_EPIC_noun_classes.csv"
)

In [ ]:
# Clip creation and processing functions

@torch.inference_mode()
def encode_clips(clips):
    """
    Encode a batch of video clips.

    Parameters
    ----------
    clips:
        List of NumPy arrays.

        Each clip has shape:
        [16, 3, H, W]

        Frames have already been processed by
        VideoMAEImageProcessor.

    Returns
    -------
    features:
        CPU tensor of shape [B, 768].
    """

    # Convert the list of clips into one batch.
    #
    # Before stacking:
    #     Each clip: [16, 3, H, W]
    #
    # After stacking:
    #     Batch:     [B, 16, 3, H, W]

    x = torch.from_numpy(np.stack(clips)).to(device=device, dtype=torch.float32)

    # Use mixed precision on CUDA to reduce memory usage.
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
        # Expected shape of last_hidden_state:
        # [B, number_of_tokens, 768]

        output = backbone(pixel_values=x)
        hidden = output.last_hidden_state

    # Average the token representations.
    #
    # [B, number_of_tokens, 768]
    #                 |
    #                 | mean(dim=1)
    #                 v
    # [B, 768]

    features = hidden.float().mean(dim=1)

    # Return the features on the CPU.
    return features.cpu()

MEAN_F = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD_F = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def preprocess_frame(frame):
    img = frame.reformat(width=SIZE, height=SIZE, format="rgb24").to_ndarray()  # [224,224,3] uint8
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0                  # [3,224,224]
    return ((x - MEAN_F) / STD_F).numpy()

def extract_clip_features(video_path, batch_clips=16, max_seconds=None):
    """
    Extract one VideoMAE feature vector per clip.

    Clips contain 16 sampled frames, sampled at
    approximately 8 FPS.

    A new clip is produced approximately every
    CLIP_STRIDE seconds, once a full clip exists.

    Parameters
    ----------
    video_path:
        Path to the video file.

    batch_clips:
        Number of clips encoded together.

    max_seconds:
        Optional maximum video duration to process.

    Returns
    -------
    clip_features:
        CPU tensor of shape [N, 768].

    clip_end_times:
        CPU tensor of shape [N].

        clip_end_times[i] records the actual timestamp
        of the last sampled frame in clip_features[i].
    """

    # --------------------------------------------------
    # 1. Open the video
    # --------------------------------------------------

    container = av.open(video_path)
    try:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"

        # --------------------------------------------------
        # 2. Establish the sampling schedule
        # --------------------------------------------------

        sample_interval = 1.0 / SAMPLE_FPS

        # First target sampling time: 0.125 seconds.
        next_sample_time = sample_interval

        # We need 16 sampled frames before a clip exists.
        #
        # At 8 FPS, 16 sampling intervals span 2 seconds.
        #
        # Subsequent clip endpoints are approximately:
        # 2, 3, 4, 5, ... seconds.

        next_clip_time = CLIP_FRAMES * sample_interval

        # --------------------------------------------------
        # 3. Buffers and output collections
        # --------------------------------------------------

        # Most recent 16 preprocessed frames.
        frame_buffer = collections.deque(maxlen=CLIP_FRAMES)

        # Actual timestamps corresponding to those frames.
        frame_time_buffer = collections.deque(maxlen=CLIP_FRAMES)

        # Clips waiting to be encoded as a batch.
        pending_clips = []

        # Timestamps of those pending clips.
        pending_times = []

        # Encoded feature batches.
        feature_batches = []

        # Timestamps aligned with the encoded features.
        clip_end_times = []

        # We will use timestamps relative to the start
        # of the video.
        origin_time = None

        # --------------------------------------------------
        # 4. Decode and sample frames
        # --------------------------------------------------

        for frame in container.decode(stream):

            TIME_EPS = 1e-6

            # Obtain the frame's timestamp.
            if frame.time is not None:
                raw_time = float(frame.time)

            elif frame.pts is not None:
                raw_time = float(frame.pts * stream.time_base)

            else:
                # Cannot place this frame in time.
                continue

            # Establish the video's time origin.
            if origin_time is None:
                origin_time = raw_time

            # Express timestamps relative to video start.
            t = raw_time - origin_time

            # Stop before processing frames beyond the
            # requested duration.
            if (max_seconds is not None and t > max_seconds + TIME_EPS):
                break

            # Ignore frames that occur before the next
            # target sampling time.
            if t < next_sample_time - TIME_EPS:
                continue

            # --------------------------------------------------
            # 5. Preprocess this sampled frame
            # --------------------------------------------------

            # Decode the frame as an RGB image.
            rgb_frame = frame.to_ndarray(format="rgb24")

            # Resize, crop and normalize according to the
            # pretrained VideoMAE processor's configuration.
            #
            # Output shape:
            # [1, 3, H, W]

            processed_frame = preprocess_frame(frame)

            # Move the preprocessed frame to a NumPy array.
            #
            # Shape:
            # [3, H, W]

            processed_frame = processed_frame

            # Add the frame and its actual timestamp.
            frame_buffer.append(processed_frame)
            frame_time_buffer.append(t)

            # --------------------------------------------------
            # 6. Advance the sampling schedule
            # --------------------------------------------------

            # We have used this decoded frame once.
            #
            # Advance past every target sampling time that
            # this frame has reached or passed. If decoding
            # skipped a sampling target, we do not duplicate
            # this frame to fill the missing timestamp.

            while (next_sample_time <= t + TIME_EPS):
                next_sample_time += sample_interval

            # --------------------------------------------------
            # 7. Check whether we can construct a clip
            # --------------------------------------------------

            # A clip requires exactly 16 sampled frames.
            if len(frame_buffer) < CLIP_FRAMES:
                continue

            # Do not create a clip until its scheduled
            # endpoint has been reached.
            if t < next_clip_time - TIME_EPS:
                continue

            # At this point:
            #
            # frame_buffer contains exactly 16 frames.
            #
            # The most recent frame has timestamp t.
            #
            # Construct one clip from the buffer.
            clip = np.stack(list(frame_buffer))

            pending_clips.append(clip)

            # Record the actual timestamp of the final
            # sampled frame in this clip.
            #
            # Do not record the scheduled timestamp:
            # the actual decoded frame may occur slightly
            # before or after it.

            actual_clip_end = frame_time_buffer[-1]
            pending_times.append(actual_clip_end)

            # --------------------------------------------------
            # 8. Advance the clip schedule
            # --------------------------------------------------

            # Advance past every clip endpoint that has
            # already been reached.
            #
            # Crucially, we created only ONE clip above.
            # We do not generate several duplicate clips
            # if a decoded frame crosses multiple endpoints.

            while (next_clip_time <= t + TIME_EPS):
                next_clip_time += CLIP_STRIDE

            # --------------------------------------------------
            # 9. Encode a batch when it is full
            # --------------------------------------------------

            if len(pending_clips) >= batch_clips:
                batch_features = encode_clips(pending_clips)
                feature_batches.append(batch_features)

                # Preserve timestamp order.
                clip_end_times.extend(pending_times)

                # Start a fresh batch.
                pending_clips = []
                pending_times = []

        # --------------------------------------------------
        # 10. Encode any remaining clips
        # --------------------------------------------------

        if pending_clips:
            batch_features = encode_clips(pending_clips)

            feature_batches.append(batch_features)

            clip_end_times.extend(pending_times)

    finally:

        # Close the video even if an error occurs.
        container.close()

    # --------------------------------------------------
    # 11. Handle videos that produced no valid clips
    # --------------------------------------------------

    if not feature_batches:

        return (torch.empty((0, FEATURE_DIM), dtype=torch.float32), torch.empty((0,),dtype=torch.float64))

    # --------------------------------------------------
    # 12. Combine batches and return aligned outputs
    # --------------------------------------------------

    clip_features = torch.cat(feature_batches, dim=0)

    clip_end_times = torch.tensor(clip_end_times, dtype=torch.float64)

    return clip_features, clip_end_times


def find_video(video_id):
    matches = glob.glob(f"{VIDEO_DIR}/**/{video_id}*.mp4", recursive=True)
    if not matches:
        raise FileNotFoundError(f"No .mp4 found for {video_id} under {VIDEO_DIR}")
    return matches[0]

In [ ]:
# Experiment

video_path = find_video("P07-20240529-102652")
clip_features, clip_end_times = extract_clip_features(video_path, batch_clips=16, max_seconds=10)
print("Feature shape:", clip_features.shape)
print("End times:", clip_end_times)
assert len(clip_features) == len(clip_end_times)

Feature shape: torch.Size([9, 768])
End times: tensor([ 2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.], dtype=torch.float64)


In [ ]:
# Noun-class mapping

import ast
import pandas as pd

noun_classes_path = f"{ROOT}/Annotations/narrations-and-action-segments/HD_EPIC_noun_classes.csv"
noun_classes_df = pd.read_csv(noun_classes_path)

if isinstance(noun_classes_df["instances"].iloc[0], str):
    noun_classes_df["instances"] = noun_classes_df["instances"].apply(ast.literal_eval)

instance_to_class = {}

for _, row in noun_classes_df.iterrows():
    class_id = int(row["id"])
    for instance_name in row["instances"]:
        if instance_name in instance_to_class:
            raise ValueError(f"Instance appears in multiple classes: {instance_name!r}")
        instance_to_class[instance_name] = class_id

# Add an unknown class for object names absent from the noun-class mapping.
NUM_KNOWN_CLASSES = len(noun_classes_df)
UNKNOWN_CLASS_ID = NUM_KNOWN_CLASSES
NUM_CLASSES = NUM_KNOWN_CLASSES + 1

def get_class_id(object_name):
    return instance_to_class.get(object_name, UNKNOWN_CLASS_ID)

print(f"Number of known noun classes: {NUM_KNOWN_CLASSES}")
print(f"Number of mapped instance names: {len(instance_to_class)}")
print(f"Unknown class ID: {UNKNOWN_CLASS_ID}")
print(f"Total classes, including unknown: {NUM_CLASSES}")

Number of known noun classes: 303
Number of mapped instance names: 19528
Unknown class ID: 303
Total classes, including unknown: 304


In [ ]:
# Label creation functions

import numpy as np


def create_labels(video_annotations, clip_end_times, video_duration, horizon=PREDICTION_HORIZON,):
    """
    Create future object-relevance labels for one video.

    Parameters
    ----------
    video_annotations:
        Object annotations for one video from assoc_info.json.

    clip_end_times:
        Actual end timestamps of the extracted feature clips.

    video_duration:
        Duration of the video in seconds.

    horizon:
        Future prediction horizon in seconds.

    Returns
    -------
    Dictionary containing:
        observation_times
        relevance
        time_to_relevance
        time_mask
    """

    # Convert the input timestamps to a NumPy array.
    clip_end_times = np.asarray(clip_end_times, dtype=np.float64,)

    # Retain only observations for which the entire future
    # prediction horizon is covered by the video.
    valid_indices = np.flatnonzero(clip_end_times + horizon <= video_duration)

    observation_times = clip_end_times[valid_indices]

    # Allocate one target row per valid observation and
    # one column per output category.
    num_observations = len(observation_times)
    relevance = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32,)
    time_to_relevance = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32,)
    time_mask = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32,)

    # Store the earliest future onset found for each
    # observation and class.
    earliest_onsets = np.full((num_observations, NUM_CLASSES), np.inf, dtype=np.float64,)

    # Examine every annotated object in this video.
    for object_id, object_info in video_annotations.items():

        # Map the object to its known class, or to unknown.
        object_name = object_info["name"]
        class_id = get_class_id(object_name)

        # Examine every temporal track belonging to the object.
        for track in object_info.get("tracks", []):

            # The first element is the track onset.
            track_start = float(track["time_segment"][0])

            # Find observations for which this onset lies
            # strictly in the future and within the horizon.
            delays = track_start - observation_times

            qualifies = ((delays > 0.0) & (delays <= horizon))

            # Find the observation rows affected by this onset.
            affected_rows = np.flatnonzero(qualifies)

            # Retain the earliest onset for each row and class.
            earliest_onsets[affected_rows, class_id] = np.minimum(earliest_onsets[affected_rows, class_id], track_start,)

    # Convert the earliest-onset information into targets.
    for observation_index, observation_time in enumerate(observation_times):

        # A finite value indicates that a qualifying future
        # track onset exists for this class.
        positive_classes = np.flatnonzero(np.isfinite(earliest_onsets[observation_index]))

        for class_id in positive_classes:
            onset_time = earliest_onsets[observation_index, class_id,]

            relevance[observation_index, class_id] = 1.0

            time_to_relevance[observation_index, class_id, ] = onset_time - observation_time

            time_mask[observation_index, class_id,] = 1.0

    return {
        "observation_times": observation_times,
        "relevance": relevance,
        "time_to_relevance": time_to_relevance,
        "time_mask": time_mask,
    }

In [ ]:
# Experiment

import av


video_id = "P07-20240529-102652"

video_path = f"{VIDEO_DIR}/P07/{video_id}.mp4"

with av.open(video_path) as container:
    video_duration = container.duration / av.time_base

print("Video ID:", video_id)
print("Video duration:", video_duration, "seconds")

video_labels = create_labels(
    video_annotations=assoc_info[video_id],
    clip_end_times=clip_end_times.numpy(),
    video_duration=video_duration,
)

print("Observation times:", video_labels["observation_times"].shape)
print("Relevance:", video_labels["relevance"].shape)
print("Time to relevance:", video_labels["time_to_relevance"].shape)
print("Time mask:", video_labels["time_mask"].shape)

# Find observation/class pairs with a future track onset.
positive_pairs = np.argwhere(
    video_labels["relevance"] == 1
)

print("Number of positive observation/class pairs:", len(positive_pairs))

for observation_index, class_id in positive_pairs[:10]:

    observation_time = video_labels["observation_times"][
        observation_index
    ]

    delay = video_labels["time_to_relevance"][
        observation_index,
        class_id,
    ]

    if class_id == UNKNOWN_CLASS_ID:
        class_name = "unknown"
    else:
        class_name = noun_classes_df.loc[
            noun_classes_df["id"] == class_id,
            "key",
        ].iloc[0]

    print(
        f"Observation: {observation_time:.2f}s | "
        f"Class: {class_name} ({class_id}) | "
        f"Time to onset: {delay:.2f}s"
    )

# Validation 1: Check relevance, time-to-relevance, and time-mask consistency

relevance = video_labels["relevance"]
time_to_relevance = video_labels["time_to_relevance"]
time_mask = video_labels["time_mask"]

assert np.array_equal(time_mask, relevance), "Time mask and relevance labels differ."
assert np.all(time_to_relevance[time_mask == 1] > 0), "A positive time target is not greater than zero."
assert np.all(time_to_relevance[time_mask == 1] <= PREDICTION_HORIZON), "A positive time target exceeds the prediction horizon."
assert np.all(time_to_relevance[time_mask == 0] == 0), "An invalid time target is not zero."

print("Validation 1 passed: relevance, time targets, and masks are consistent.")

# Validation 2: Inspect the unknown-class onset annotations

unknown_observations = np.flatnonzero(relevance[:, UNKNOWN_CLASS_ID] == 1)

for observation_index in unknown_observations:
    observation_time = video_labels["observation_times"][observation_index]
    delay = time_to_relevance[observation_index, UNKNOWN_CLASS_ID]
    expected_onset = observation_time + delay
    print(f"Observation: {observation_time:.2f}s | Expected earliest unknown onset: {expected_onset:.2f}s | Delay: {delay:.2f}s")

    for object_id, object_info in assoc_info[video_id].items():
        if get_class_id(object_info["name"]) != UNKNOWN_CLASS_ID:
            continue
        for track in object_info.get("tracks", []):
            track_start = float(track["time_segment"][0])
            if np.isclose(track_start, expected_onset, atol=1e-5):
                print(f"  Matching object: {object_info['name']!r} | Track start: {track_start:.2f}s")

# Validation 3: Compare annotation timestamps with video timestamps

import av

video_path = f"{VIDEO_DIR}/{video_id[:3]}/{video_id}.mp4"

with av.open(video_path) as container:
    video_duration = container.duration / av.time_base
    video_stream = next(stream for stream in container.streams if stream.type == "video")
    stream_duration = float(video_stream.duration * video_stream.time_base) if video_stream.duration is not None else None

annotation_starts = [float(track["time_segment"][0]) for obj in assoc_info[video_id].values() for track in obj.get("tracks", [])]
annotation_ends = [float(track["time_segment"][1]) for obj in assoc_info[video_id].values() for track in obj.get("tracks", [])]

print(f"Video duration (container): {video_duration:.2f}s")
print(f"Video duration (stream): {stream_duration:.2f}s" if stream_duration is not None else "Video stream duration unavailable.")
print(f"Earliest annotated track start: {min(annotation_starts):.2f}s" if annotation_starts else "No annotated track starts.")
print(f"Latest annotated track end: {max(annotation_ends):.2f}s" if annotation_ends else "No annotated track ends.")
print(f"First extracted clip endpoint: {float(clip_end_times[0]):.2f}s")
print(f"Last extracted clip endpoint: {float(clip_end_times[-1]):.2f}s")

assert all(start >= 0 for start in annotation_starts), "An annotation starts before time zero."
assert all(end >= start for start, end in zip(annotation_starts, annotation_ends)), "An annotation ends before it starts."
assert not annotation_ends or max(annotation_ends) <= video_duration + 1.0, "Annotation times extend substantially beyond the video duration."

print("Basic timestamp checks passed. Confirm that annotation and video timestamps share the same time origin.")

# Verify annotation timestamps against actual video-frame timestamps

import av
import numpy as np

video_id = "P07-20240529-102652"
video_path = f"{VIDEO_DIR}/{video_id[:3]}/{video_id}.mp4"
video_annotations = assoc_info[video_id]
video_masks = mask_info[video_id]

# Select the earliest track and the previously identified mug2 track.
tracks_to_check = []

for object_id, object_info in video_annotations.items():
    for track in object_info.get("tracks", []):
        start, end = map(float, track["time_segment"])
        if np.isclose(start, 7.05, atol=0.02) or (object_info["name"] == "mug2" and np.isclose(start, 14.05, atol=0.02)):
            mask_ids = track.get("masks", [])
            mask_frames = [(mask_id, int(video_masks[mask_id]["frame_number"])) for mask_id in mask_ids if mask_id in video_masks]
            tracks_to_check.append({"name": object_info["name"], "start": start, "end": end, "mask_frames": mask_frames})

if not tracks_to_check:
    raise ValueError("No matching tracks found. Check the annotation structure and timestamps.")

target_frames = sorted({frame_number for track in tracks_to_check for _, frame_number in track["mask_frames"]})

if not target_frames:
    raise ValueError("No linked mask frames found for the selected tracks.")

# Decode only as far as the latest target frame, recording actual frame timestamps.
decoded_times = {}

with av.open(video_path) as container:
    stream = container.streams.video[0]
    fps = float(stream.average_rate) if stream.average_rate else None
    print(f"Video FPS (average rate): {fps}")
    for frame_number, frame in enumerate(container.decode(video=0)):
        if frame_number in target_frames:
            decoded_times[frame_number] = float(frame.pts * stream.time_base) if frame.pts is not None else None
        if frame_number >= target_frames[-1]:
            break

# Compare annotated intervals with timestamps of their linked mask frames.
for track in tracks_to_check:
    print(f"\nObject: {track['name']!r}")
    print(f"Annotated interval: {track['start']:.3f}s - {track['end']:.3f}s")
    if not track["mask_frames"]:
        print("No linked masks available.")
        continue
    for mask_id, frame_number in track["mask_frames"]:
        frame_time = decoded_times.get(frame_number)
        if frame_time is None:
            print(f"  Mask {mask_id}: frame {frame_number}; timestamp unavailable")
            continue
        start_difference = frame_time - track["start"]
        end_difference = frame_time - track["end"]
        print(f"  Mask {mask_id}: frame {frame_number}, video time {frame_time:.3f}s, difference from start {start_difference:+.3f}s, difference from end {end_difference:+.3f}s")

Video ID: P07-20240529-102652
Video duration: 931.5 seconds
Observation times: (9,)
Relevance: (9, 304)
Time to relevance: (9, 304)
Time mask: (9, 304)
Number of positive observation/class pairs: 28
Observation: 2.00s | Class: bowl (7) | Time to onset: 5.05s
Observation: 2.00s | Class: cup (13) | Time to onset: 10.55s
Observation: 2.00s | Class: unknown (303) | Time to onset: 12.06s
Observation: 3.00s | Class: bowl (7) | Time to onset: 4.05s
Observation: 3.00s | Class: cup (13) | Time to onset: 9.55s
Observation: 3.00s | Class: unknown (303) | Time to onset: 11.06s
Observation: 4.00s | Class: bowl (7) | Time to onset: 3.05s
Observation: 4.00s | Class: cup (13) | Time to onset: 8.55s
Observation: 4.00s | Class: unknown (303) | Time to onset: 10.06s
Observation: 5.00s | Class: bowl (7) | Time to onset: 2.05s
Validation 1 passed: relevance, time targets, and masks are consistent.
Observation: 2.00s | Expected earliest unknown onset: 14.06s | Delay: 12.06s
  Matching object: 'mug2' | Track

In [ ]:
# Label storing functions

# Dataset configuration

VIDEO_DIR = f"{ROOT}/Videos"
DATASET_DIR = f"{ROOT}/Dataset"
VIDEO_DATA_DIR = f"{DATASET_DIR}/videos"
PREDICTION_HORIZON = 30.0
NUM_KNOWN_CLASSES = len(noun_classes_df)
UNKNOWN_CLASS_ID = NUM_KNOWN_CLASSES
NUM_CLASSES = NUM_KNOWN_CLASSES + 1
FEATURE_DIM = 768

os.makedirs(VIDEO_DATA_DIR, exist_ok=True)
print(f"Known classes: {NUM_KNOWN_CLASSES} | Unknown class: {UNKNOWN_CLASS_ID} | Total classes: {NUM_CLASSES}")

# Label-generation function

def create_labels(video_annotations, clip_end_times, video_duration, horizon=PREDICTION_HORIZON):
    clip_end_times = np.asarray(clip_end_times, dtype=np.float64)
    valid_indices = np.flatnonzero(clip_end_times + horizon <= video_duration)
    observation_times = clip_end_times[valid_indices]
    num_observations = len(observation_times)
    relevance = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32)
    time_to_relevance = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32)
    time_mask = np.zeros((num_observations, NUM_CLASSES), dtype=np.float32)
    earliest_onsets = np.full((num_observations, NUM_CLASSES), np.inf, dtype=np.float64)

    for object_info in video_annotations.values():
        class_id = get_class_id(object_info["name"])
        for track in object_info.get("tracks", []):
            track_start = float(track["time_segment"][0])
            affected_rows = np.flatnonzero((track_start > observation_times) & (track_start <= observation_times + horizon))
            earliest_onsets[affected_rows, class_id] = np.minimum(earliest_onsets[affected_rows, class_id], track_start)

    for i, observation_time in enumerate(observation_times):
        positive_classes = np.flatnonzero(np.isfinite(earliest_onsets[i]))
        relevance[i, positive_classes] = 1.0
        time_to_relevance[i, positive_classes] = earliest_onsets[i, positive_classes] - observation_time
        time_mask[i, positive_classes] = 1.0

    return {"observation_times": observation_times, "relevance": relevance, "time_to_relevance": time_to_relevance, "time_mask": time_mask, "valid_feature_indices": valid_indices}

def find_video(video_id):
    matches = sorted(Path(VIDEO_DIR).rglob(f"{video_id}.mp4"))
    if not matches:
        return None
    if len(matches) > 1:
        raise ValueError(f"Multiple video files found for {video_id}: {matches}")
    return str(matches[0])

available_video_ids = sorted(video_id for video_id in assoc_info if find_video(video_id) is not None)
print(f"Annotated videos available locally: {len(available_video_ids)}")

processing_errors = []

for index, video_id in enumerate(available_video_ids, start=1):
    save_path = f"{VIDEO_DATA_DIR}/{video_id}.npz"

    if os.path.exists(save_path):
        print(f"Already saved: {video_id}")
        continue

    if os.path.exists(save_path):
        print(f"[{index}/{len(available_video_ids)}] Already saved: {video_id}")
        continue

    try:
        video_path = find_video(video_id)

        with av.open(video_path) as container:
            if container.duration is not None:
                video_duration = container.duration / av.time_base
            else:
                stream = container.streams.video[0]
                if stream.duration is None:
                    raise ValueError("Could not determine video duration.")
                video_duration = float(stream.duration * stream.time_base)

        clip_features, clip_end_times = extract_clip_features(video_path, batch_clips=16, max_seconds=None)
        clip_end_times = clip_end_times.cpu().numpy() if hasattr(clip_end_times, "cpu") else np.asarray(clip_end_times)
        clip_features = clip_features.cpu().numpy() if hasattr(clip_features, "cpu") else np.asarray(clip_features)

        labels = create_labels(assoc_info[video_id], clip_end_times, video_duration)
        valid_indices = labels["valid_feature_indices"]
        features = clip_features[valid_indices]

        assert features.shape == (len(labels["observation_times"]), FEATURE_DIM)
        assert np.array_equal(labels["time_mask"], labels["relevance"])
        assert np.all(labels["time_to_relevance"][labels["time_mask"] == 1] > 0)
        assert np.all(labels["time_to_relevance"][labels["time_mask"] == 1] <= PREDICTION_HORIZON)

        np.savez_compressed(save_path, features=features, observation_times=labels["observation_times"], relevance=labels["relevance"], time_to_relevance=labels["time_to_relevance"], time_mask=labels["time_mask"], video_duration=np.float64(video_duration))

        print(f"[{index}/{len(available_video_ids)}] Saved {video_id}: {len(features)} observations")

    except Exception as error:
        processing_errors.append((video_id, repr(error)))
        print(f"[{index}/{len(available_video_ids)}] FAILED {video_id}: {error}")

print(f"\nVideos available: {len(available_video_ids)}")
print(f"Errors: {len(processing_errors)}")
if processing_errors:
    print(*processing_errors, sep="\n")

saved_video_ids = sorted(path.stem for path in Path(VIDEO_DATA_DIR).glob("*.npz"))
total_saved_observations = 0

for video_id in saved_video_ids:
    with np.load(f"{VIDEO_DATA_DIR}/{video_id}.npz") as data:
        n = len(data["observation_times"])
        assert data["features"].shape == (n, FEATURE_DIM), f"Feature mismatch: {video_id}"
        assert data["relevance"].shape == (n, NUM_CLASSES), f"Relevance mismatch: {video_id}"
        assert data["time_to_relevance"].shape == (n, NUM_CLASSES), f"Time-target mismatch: {video_id}"
        assert data["time_mask"].shape == (n, NUM_CLASSES), f"Mask mismatch: {video_id}"
        assert np.array_equal(data["time_mask"], data["relevance"]), f"Mask mismatch: {video_id}"
        assert np.all(data["time_to_relevance"][data["time_mask"] == 1] > 0), f"Invalid time target: {video_id}"
        assert np.all(data["time_to_relevance"][data["time_mask"] == 1] <= PREDICTION_HORIZON), f"Time target exceeds horizon: {video_id}"
        total_saved_observations += n

print(f"Verified {len(saved_video_ids)} saved videos.")
print(f"Total observations: {total_saved_observations:,}")

existing_manifest_paths = [f"{DATASET_DIR}/split_manifest.json", f"{ROOT}/split_manifest.json", f"{ROOT}/Dataset/split_manifest.json"]
existing_manifest_path = next((p for p in existing_manifest_paths if os.path.exists(p)), None)

if existing_manifest_path:
    with open(existing_manifest_path) as file:
        split_manifest = json.load(file)
    print(f"Found existing split manifest: {existing_manifest_path}")
else:
    print("No existing split manifest found in the checked locations.")

if existing_manifest_path is None:
    if len(saved_video_ids) != 34:
        raise ValueError(f"Expected 34 saved videos for a 23/6/5 split, but found {len(saved_video_ids)}. Check processing before splitting.")

    shuffled_video_ids = saved_video_ids.copy()
    random.Random(42).shuffle(shuffled_video_ids)

    split_manifest = {"train": shuffled_video_ids[:23], "validation": shuffled_video_ids[23:29], "test": shuffled_video_ids[29:34]}

train_video_ids = split_manifest["train"]
val_video_ids = split_manifest["validation"]
test_video_ids = split_manifest["test"]

train_set, val_set, test_set = map(set, (train_video_ids, val_video_ids, test_video_ids))
saved_set = set(saved_video_ids)

assert len(train_set) == len(train_video_ids), "Duplicate training IDs."
assert len(val_set) == len(val_video_ids), "Duplicate validation IDs."
assert len(test_set) == len(test_video_ids), "Duplicate test IDs."
assert not (train_set & val_set or train_set & test_set or val_set & test_set), "Video leakage between splits."
assert train_set | val_set | test_set == saved_set, "Some saved videos are missing from the split or the manifest includes unavailable videos."

split_path = f"{DATASET_DIR}/split_manifest.json"
with open(split_path, "w") as file:
    json.dump(split_manifest, file, indent=2)

print(f"Training videos: {len(train_video_ids)}")
print(f"Validation videos: {len(val_video_ids)}")
print(f"Test videos: {len(test_video_ids)}")
print(f"Saved split manifest: {split_path}")

for split_name, video_ids in [("Train", train_video_ids), ("Validation", val_video_ids), ("Test", test_video_ids)]:
    total_observations = 0
    positive_pairs = 0

    for video_id in video_ids:
        with np.load(f"{VIDEO_DATA_DIR}/{video_id}.npz") as data:
            total_observations += len(data["observation_times"])
            positive_pairs += int(data["relevance"].sum())

    print(f"{split_name}: {len(video_ids)} videos | {total_observations:,} observations | {positive_pairs:,} positive observation/class pairs")

Known classes: 303 | Unknown class: 303 | Total classes: 304
Annotated videos available locally: 34
[1/34] Saved P07-20240529-102652: 900 observations
[2/34] Saved P07-20240529-131737: 576 observations
[3/34] Saved P07-20240529-134410: 1033 observations
[4/34] Saved P07-20240529-191007: 1879 observations
[5/34] Saved P07-20240529-194518: 1671 observations
[6/34] Saved P07-20240530-102543: 2065 observations
[7/34] Saved P07-20240530-111344: 128 observations
[8/34] Saved P07-20240531-101357: 2548 observations
[9/34] Saved P07-20240531-201703: 1852 observations
[10/34] Saved P08-20240613-122900: 167 observations
[11/34] Saved P08-20240613-123547: 980 observations
[12/34] Saved P08-20240614-085000: 1056 observations
[13/34] Saved P08-20240614-125650: 1193 observations
[14/34] Saved P08-20240614-132938: 127 observations
[15/34] Saved P08-20240614-184945: 977 observations
[16/34] Saved P08-20240617-130401: 1962 observations
[17/34] Saved P08-20240617-184909: 1841 observations
[18/34] Saved P